In [4]:
# Import core libraries
import pandas as pd
import numpy as np

In [6]:
# Load dataset
df = pd.read_csv('Readmissions_and_Deaths_-_Hospital.csv')

# Preview the data
df.head()

,index,Provider ID,Hospital Name,Address,City,State,ZIP Code,County Name,Phone Number,Measure Name,Measure ID,Compared to National,Denominator,Score,Lower Estimate,Higher Estimate,Footnote,Measure Start Date,Measure End Date,Location
0,0,230100,TAWAS ST JOSEPH HOSPITAL,200 HEMLOCK,TAWAS CITY,MI,48764,IOSCO,9893629301,Rate of readmission after discharge from hospi...,READM_30_HOSP_WIDE,No Different than the National Rate,438,13.9,12.6,15.6,NaN,07/01/2014,06/30/2015,"200 HEMLOCK\nTAWAS CITY, MI 48764\n(44.274911,..."
1,1,230121,MEMORIAL HEALTHCARE,826 WEST KING STREET,OWOSSO,MI,48867,SHIAWASSEE,9897235211,Rate of readmission after hip/knee replacement,READM_30_HIP_KNEE,No Different than the National Rate,150,4.0,2.8,5.7,NaN,07/01/2012,06/30/2015,"826 WEST KING STREET\nOWOSSO, MI 48867\n(43.00..."
2,2,230118,HURON MEDICAL CENTER,1100 SOUTH VAN DYKE ROAD,BAD AXE,MI,48413,HURON,9892699521,Pneumonia (PN) 30-Day Readmission Rate,READM_30_PN,No Different than the National Rate,205,16.7,13.8,19.9,NaN,07/01/2012,06/30/2015,"1100 SOUTH VAN DYKE ROAD\nBAD AXE, MI 48413\n(..."
3,3,230121,MEMORIAL HEALTHCARE,826 WEST KING STREET,OWOSSO,MI,48867,SHIAWASSEE,9897235211,Rate of readmission for stroke patients,READM_30_STK,No Different than the National Rate,67,11.5,8.7,14.9,NaN,07/01/2012,06/30/2015,"826 WEST KING STREET\nOWOSSO, MI 48867\n(43.00..."
4,4,230133,OTSEGO MEMORIAL HOSPITAL,825 N CENTER AVE,GAYLORD,MI,49735,OTSEGO,9897312100,Heart failure (HF) 30-Day Mortality Rate,MORT_30_HF,No Different than the National Rate,102,13.4,9.9,17.7,NaN,07/01/2012,06/30/2015,"825 N CENTER AVE\nGAYLORD, MI 49735\n(45.03537..."


In [17]:
# Convert Score and Denominator to numeric
df['Score'] = pd.to_numeric(df['Score'], errors='coerce')
df['Denominator'] = pd.to_numeric(df['Denominator'], errors='coerce')

In [7]:
# Check columns and data types
df.info()

# Basic statistics
df.describe()

# View column names clearly
df.columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64764 entries, 0 to 64763
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   index                 64764 non-null  int64 
 1   Provider ID           64764 non-null  int64 
 2   Hospital Name         64764 non-null  object
 3   Address               64764 non-null  object
 4   City                  64764 non-null  object
 5   State                 64764 non-null  object
 6   ZIP Code              64764 non-null  int64 
 7   County Name           64582 non-null  object
 8   Phone Number          64764 non-null  int64 
 9   Measure Name          64764 non-null  object
 10  Measure ID            64764 non-null  object
 11  Compared to National  64764 non-null  object
 12  Denominator           64764 non-null  object
 13  Score                 64764 non-null  object
 14  Lower Estimate        64764 non-null  object
 15  Higher Estimate       64764 non-null

Index(['index', 'Provider ID', 'Hospital Name', 'Address', 'City', 'State',
       'ZIP Code', 'County Name', 'Phone Number', 'Measure Name', 'Measure ID',
       'Compared to National', 'Denominator', 'Score', 'Lower Estimate',
       'Higher Estimate', 'Footnote', 'Measure Start Date', 'Measure End Date',
       'Location'],
      dtype='object')

In [10]:
# Check missing values in score column
df['Score'].isna().sum()

# Keep only rows with a numeric score
df_score = df[df['Score'].notna()]

In [11]:
# Total records
total_records = len(df)

# Records with scores
scored_records = len(df_score)

# Percentage
percentage = (scored_records / total_records) * 100

total_records, scored_records, percentage

(64764, 64764, 100.0)

In [13]:
# Count categories
comparison_counts = df['Compared to National'].value_counts()

comparison_counts

Compared to National
No Different than the National Rate    39360
Not Available                          11779
Number of Cases Too Small              11200
Worse than the National Rate            1350
Better than the National Rate           1075
Name: count, dtype: int64

In [16]:
valid_comparisons = df[
    ~df['Compared to National'].isin([
        'Not Available',
        'Number of Cases Too Small'
    ])
]

In [18]:
# Total records
total_records = len(df)

# Keep only rows with valid numeric score
df_score = df[df['Score'].notna()]

# Count
scored_records = len(df_score)

# Percentage
percentage = (scored_records / total_records) * 100

total_records, scored_records, percentage

(64764, 41785, 64.51886850719536)

In [19]:
# Count each category
comparison_counts = df['Compared to National'].value_counts()

comparison_counts

Compared to National
No Different than the National Rate    39360
Not Available                          11779
Number of Cases Too Small              11200
Worse than the National Rate            1350
Better than the National Rate           1075
Name: count, dtype: int64

In [20]:
# Specific categories
no_diff = (df['Compared to National'] == 'No Different than the National Rate').sum()
worse = (df['Compared to National'] == 'Worse than the National Rate').sum()
better = (df['Compared to National'] == 'Better than the National Rate').sum()

no_diff, worse, better

(np.int64(39360), np.int64(1350), np.int64(1075))

In [21]:
invalid = df[
    df['Compared to National'].isin([
        'Not Available',
        'Number of Cases Too Small'
    ])
]

len(invalid)

22979

In [22]:
df_valid = df[
    (df['Score'].notna()) &
    (~df['Compared to National'].isin([
        'Not Available',
        'Number of Cases Too Small'
    ]))
]

In [31]:
# Group by measure and calculate mean score
readmit_avg = df_readmit.groupby('Measure Name')['Score'].mean().sort_values(ascending=False)

measure_avg.head(10)

Measure Name
Heart failure (HF) 30-Day Readmission Rate                                       21.956284
Rate of readmission for chronic obstructive pulmonary disease (COPD) patients    20.000027
Pneumonia (PN) 30-Day Readmission Rate                                           17.113525
Acute Myocardial Infarction (AMI) 30-Day Readmission Rate                        16.891459
Pneumonia (PN) 30-Day Mortality Rate                                             16.399121
Rate of readmission after discharge from hospital (hospital-wide)                15.578305
Death rate for stroke patients                                                   14.929153
Rate of readmission for CABG                                                     14.408171
Acute Myocardial Infarction (AMI) 30-Day Mortality Rate                          14.063933
Rate of readmission for stroke patients                                          12.566986
Name: Score, dtype: float64

In [24]:
df['Measure Name'].unique()

array(['Rate of readmission after discharge from hospital (hospital-wide)',
       'Rate of readmission after hip/knee replacement',
       'Pneumonia (PN) 30-Day Readmission Rate',
       'Rate of readmission for stroke patients',
       'Heart failure (HF) 30-Day Mortality Rate',
       'Pneumonia (PN) 30-Day Mortality Rate',
       'Death rate for chronic obstructive pulmonary disease (COPD) patients',
       'Heart failure (HF) 30-Day Readmission Rate',
       'Acute Myocardial Infarction (AMI) 30-Day Readmission Rate',
       'Acute Myocardial Infarction (AMI) 30-Day Mortality Rate',
       'Death rate for CABG', 'Death rate for stroke patients',
       'Rate of readmission for CABG',
       'Rate of readmission for chronic obstructive pulmonary disease (COPD) patients'],
      dtype=object)

In [25]:
# Keep only readmission measures
df_readmit = df_valid[
    df_valid['Measure Name'].str.contains('Readmission', case=False)
]

df_readmit['Measure Name'].unique()

array(['Rate of readmission after discharge from hospital (hospital-wide)',
       'Rate of readmission after hip/knee replacement',
       'Pneumonia (PN) 30-Day Readmission Rate',
       'Rate of readmission for stroke patients',
       'Heart failure (HF) 30-Day Readmission Rate',
       'Acute Myocardial Infarction (AMI) 30-Day Readmission Rate',
       'Rate of readmission for chronic obstructive pulmonary disease (COPD) patients',
       'Rate of readmission for CABG'], dtype=object)

In [26]:
df_readmit_25 = df_readmit[df_readmit['Denominator'] >= 25]

In [27]:
# Sort by score descending
top_readmissions = df_readmit_25.sort_values(by='Score', ascending=False)

# View top rows
top_readmissions[['Hospital Name', 'Measure Name', 'Score', 'Denominator']].head(10)

,Hospital Name,Measure Name,Score,Denominator
42134,HARLAN ARH HOSPITAL,Heart failure (HF) 30-Day Readmission Rate,31.3,263.0
11586,CORNING HOSPITAL,Heart failure (HF) 30-Day Readmission Rate,27.4,310.0
10622,KINGS COUNTY HOSPITAL CENTER,Heart failure (HF) 30-Day Readmission Rate,27.2,152.0
11472,BETH ISRAEL MEDICAL CENTER,Heart failure (HF) 30-Day Readmission Rate,27.2,1009.0
31169,AVENTURA HOSPITAL AND MEDICAL CENTER,Heart failure (HF) 30-Day Readmission Rate,27.0,772.0
62,BEAUMONT HOSPITAL - WAYNE,Heart failure (HF) 30-Day Readmission Rate,26.9,336.0
12354,UNIVERSITY HOSPITAL OF BROOKLYN ( DOWNSTATE ),Heart failure (HF) 30-Day Readmission Rate,26.8,451.0
11042,KINGSBROOK JEWISH MEDICAL CENTER,Heart failure (HF) 30-Day Readmission Rate,26.8,247.0
15161,FISHER-TITUS HOSPITAL,Heart failure (HF) 30-Day Readmission Rate,26.8,222.0
5356,SSM HEALTH ST. MARY'S HOSPITAL - JEFFERSON CITY,Heart failure (HF) 30-Day Readmission Rate,26.8,437.0


In [28]:
top_readmissions[['Hospital Name', 'Measure Name', 'Score', 'Denominator']].head(1)

,Hospital Name,Measure Name,Score,Denominator
42134,HARLAN ARH HOSPITAL,Heart failure (HF) 30-Day Readmission Rate,31.3,263.0


In [29]:
top_readmissions.to_csv('high_readmission_summary.csv', index=False)

In [30]:
#Highest average readmission score measures

readmit_avg = df_readmit.groupby('Measure Name')['Score'].mean().sort_values(ascending=False)

readmit_avg

Measure Name
Heart failure (HF) 30-Day Readmission Rate                                       21.956284
Rate of readmission for chronic obstructive pulmonary disease (COPD) patients    20.000027
Pneumonia (PN) 30-Day Readmission Rate                                           17.113525
Acute Myocardial Infarction (AMI) 30-Day Readmission Rate                        16.891459
Rate of readmission after discharge from hospital (hospital-wide)                15.578305
Rate of readmission for CABG                                                     14.408171
Rate of readmission for stroke patients                                          12.566986
Rate of readmission after hip/knee replacement                                    4.610965
Name: Score, dtype: float64

In [ ]:
# # **Hospital Readmissions and Mortality: Descriptive Findings**
#
# The source contains 64,764 reported hospital-measure records across 4,434 hospitals,
# 55 states or territories, and 14 measures. A numeric score was available for 41,785
# records (64.5%).
#
# ## **National Comparison**
#
# Most records (39,360) were classified as **No Different than the National Rate**.
# There were 1,350 **Worse than the National Rate** records and 1,075 **Better than
# the National Rate** records. Another 22,979 records were unavailable or had too few
# cases, so they should not be treated as neutral results.
#
# ## **Highest Average Readmission Score Measures**
#
# The largest mean scores were for heart-failure readmission (21.96), COPD readmission
# (20.00), pneumonia readmission (17.11), and AMI readmission (16.89). These figures
# have different clinical definitions and should not be compared as a single quality
# ranking; they are most useful within the same measure.
#
# ## **Readmission Records with the Highest Rates**
#
# After retaining measures identified as readmissions and records with at least 25 cases,
# the highest observed score was 31.3 for Harlan ARH Hospital's heart-failure 30-day
# readmission measure (263 cases). The complete ranked list is in
# `high_readmission_summary.csv`.
#
# ## **Important Interpretation Limits**
#
# These are descriptive results, not causal quality ratings. CMS scores are measure-
# specific and may reflect case mix, eligibility, reporting periods, and statistical
# adjustment. State comparisons include territories and small samples, so do not use
# raw state means as definitive rankings.